# Pamoka 05 - Agentinis RAG


## Nustatymas

Ši užrašų knygutė demonstruoja Agentic RAG (Retrieval-Augmented Generation) šabloną, naudojant Microsoft Agent Framework.

**Išankstiniai reikalavimai:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — jūsų Azure AI Search paslaugos galinis taškas
- `AZURE_SEARCH_API_KEY` — jūsų Azure AI Search API raktas
- Azure OpenAI diegimas sukonfigūruotas per aplinkos kintamuosius
- Azure CLI autentifikuotas (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kas yra Agentic RAG?

Tradiciškai RAG veikia pagal fiksuotą procesą: pirmiausia surenkami dokumentai, o tada generuojamas atsakymas. **Agentic RAG** eina toliau, suteikdamas agentui autonomiją nuspręsti, **kada** ir **kaip** rinkti informaciją.

Su Agentic RAG agentas gali:
- **Nuspręsti**, ar prieš atsakant į klausimą reikia rinkti informaciją
- **Pasirinkti**, kurį duomenų šaltinį ar įrankį užklausti
- **Įvertinti** surinktus rezultatus ir atlikti papildomą informacijos rinkimą, jei pirmasis bandymas nepakankamas
- **Derinti** informaciją iš kelių rinkimo etapų į nuoseklų atsakymą

Tai daro agentą lankstesnį ir tikslesnį, palyginti su statiniu rink-paskui-generuok procesu.


## Paieškos įrankio kūrimas

Agentic RAG išoriniai duomenų šaltiniai apgaubti kaip **įrankiai**, kuriuos agentas gali iškvieti pagal poreikį. Tai leidžia agentui laikyti paiešką tiesiog dar viena veiksmų rūšimi, o ne privalomu žingsniu.

Žemiau mes apibrėžiame kelionių žinių bazę ir pateikiame ją kaip įrankį, kurį agentas gali iškviesti norėdamas sužinoti informaciją apie kelionės tikslą.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG agento kūrimas

Dabar kuriame agentą, kuriam nurodyta **visada prieš atsakant gauti informaciją**. Agentas naudoja įrankį `search_travel_knowledge`, kad pagrįstų savo atsakymus žinių baze, o ne remtųsi savo mokymo duomenimis.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratyvus gavimas — Maker-Checker modelis

Viena iš Agentic RAG pagrindinių privalumų yra **iteratyvus gavimas**. Agentas gali atlikti kelis paieškos etapus, kad patikrintų, patobulintų ar išplėstų savo pradinius radinius — panašiai kaip „maker-checker“ darbo eiga:

1. **Maker etapas**: Agentas gauna pradinę informaciją ir rengia atsakymą.
2. **Checker etapas**: Agentas atlieka papildomas paieškas, kad patikrintų detales arba užpildytų spragas.

Žemiau agentui pateiktas klausimas, kuriam reikia palyginti kelias paskirties vietas, todėl jis verčiamas ieškoti kelis kartus.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Santrauka

Šioje pamokoje išmokote, kaip sukurti **Agentic RAG** sistemą, naudojant Microsoft Agent Framework:

- **Agentic RAG** leidžia agentams savarankiškai nuspręsti, kada gauti informaciją, todėl informacijos gavimas tampa dinamiškas, o ne fiksuotas.
- **Įrankiai kaip duomenų šaltiniai**: Išorinės žinių bazės (pvz., Azure AI Search) yra įvyniotos į įrankius, kuriuos agentas gali iškviesti.
- **Iteratyvus gavimas**: maker-checker modelis leidžia agentui atlikti kelis informacijos gavimo etapus — ieškoti, tikrinti ir tobulinti — prieš pateikiant galutinį atsakymą.

Produkcijoje jūs pakeistumėte atmintyje esančią `TRAVEL_KNOWLEDGE_BASE` tikru Azure AI Search indeksu, kad būtų galima tvarkyti didelio masto kelionių dokumentų gavimą.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:  
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turi būti laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogišką vertimą. Mes neatsakome už bet kokius nesusipratimus ar neteisingus interpretavimus, kylančius naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
